# Notebook 8 — the macroatom

Level-1 code: the atom as a Markov process over its internal states. From level $i$, de-activate through line $i\to j$ with weight $A_{ij}\beta_{ij}(\epsilon_i-\epsilon_j)$ or jump to $j$ with weight $A_{ij}\beta_{ij}\epsilon_j$; one packet of the full energy leaves at de-activation. Compared with the explicit cascade of notebook 6.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom, H_ERG_S
rng = np.random.default_rng(rtedu.SEEDS["ch08"])
atom = five_level_atom()
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY        # the book's standard state from here on
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nm = 1e7 * atom.lam_cm

In [ ]:
from rtedu.sobolev import escape_probability
E_lev = atom.E                                       # epsilon_i, the level energies
def macro_probabilities(level, beta):
    k = np.flatnonzero(atom.upper == level)
    ab = atom.A[k] * beta[k]
    w_de = ab * (E_lev[level] - E_lev[atom.lower[k]])          # de-activate: the line's energy
    w_jp = ab * E_lev[atom.lower[k]]                            # jump: the energy that stays inside
    tot = w_de.sum() + w_jp.sum()
    return k, w_de / tot, w_jp / tot

def macro_walk(rng, level, beta):
    jumps = 0
    while True:
        k, p_de, p_jp = macro_probabilities(level, beta)
        c = np.cumsum(np.concatenate([p_de, p_jp])); i = int(np.searchsorted(c, rng.random()))
        if i < k.size:
            return int(k[i]), jumps                              # de-activation: the exit line
        level = int(atom.lower[k[i - k.size]]); jumps += 1

beta_one = np.ones(atom.n_lines)
for level in (4, 3, 2):
    k, p_de, p_jp = macro_probabilities(level, beta_one)
    print(f"from E{level}: de-activate " + ", ".join(f"{nm[j]:.0f}nm {p:.3f}" for j, p in zip(k, p_de)) + " | jump " + ", ".join(f"E{atom.lower[j]} {p:.3f}" for j, p in zip(k, p_jp)))

In [ ]:
n = 20_000
exits = np.empty(n, int); jumps = np.empty(n, int)
for i in range(n):
    exits[i], jumps[i] = macro_walk(rng, 4, beta_one)
E_line_macro = np.bincount(exits, minlength=atom.n_lines) / n          # each exit carries the full energy E = 1
mean_jumps = float(jumps.mean())
# the explicit cascade's energy per line (notebook 6), same normalisation
E4 = atom.E[4]; E_line_casc = np.zeros(atom.n_lines)
for _ in range(n):
    for j in atom.cascade(rng, 4):
        E_line_casc[j] += (atom.E[atom.upper[j]] - atom.E[atom.lower[j]]) / E4 / n
max_dev = float(np.abs(E_line_macro - E_line_casc).max())
print(f"macroatom: {mean_jumps:.2f} internal jumps per activation; max |macro - cascade| energy per line = {max_dev:.4f}")

## β once: a trapped line

Give the 769 nm line ($4\to2$) an escape probability of 0.05, as if it were very optically thick. The macroatom leaves through it less often; the energy goes elsewhere; the total is still one.

In [ ]:
beta_trap = beta_one.copy(); beta_trap[2] = 0.05
exits_t = np.array([macro_walk(rng, 4, beta_trap)[0] for _ in range(n)])
E_line_trap = np.bincount(exits_t, minlength=atom.n_lines) / n
print(f"769 nm: free {E_line_macro[2]:.3f} -> trapped {E_line_trap[2]:.3f}; total {E_line_trap.sum():.3f}")

## Validation against `rtedu`

In [ ]:
from rtedu.macroatom import ToyMacroAtom, cascade_energy_per_line
m = ToyMacroAtom(atom)
k, p_de, p_jp = m.probabilities(4); k2, q_de, q_jp = macro_probabilities(4, beta_one)
assert np.allclose(p_de, q_de) and np.allclose(p_jp, q_jp)
e_ref = m.energy_per_line(np.random.default_rng(2), 4, 20_000)
assert np.abs(e_ref - E_line_macro).max() < 0.02
print("rtedu.macroatom agrees:", np.round(e_ref, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(atom.n_lines); w_ = 0.27
axes[0].bar(x - w_, E_line_casc, w_, color=OI["orange"], label="explicit cascade (many photons)")
axes[0].bar(x, E_line_macro, w_, color=OI["blue"], label="macroatom (one energy packet)")
axes[0].bar(x + w_, E_line_trap, w_, color=OI["red"], label=r"macroatom, 769 nm trapped ($\beta$ = 0.05)")
axes[0].set_xticks(x); axes[0].set_xticklabels([f"{v:.0f}" for v in nm], rotation=60, fontsize=7); axes[0].set_ylabel("energy per activation / E"); axes[0].legend(fontsize=7)
axes[0].set_title("energy emitted per line from an activation at $E_4$", fontsize=9)
axes[1].hist(jumps, bins=np.arange(-0.5, jumps.max() + 1.5), color=OI["blue"], rwidth=0.8); axes[1].set_xlabel("internal jumps before de-activation"); axes[1].set_ylabel("activations")
axes[1].set_title(f"mean {mean_jumps:.2f} jumps", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch08_macroatom")

In [ ]:
results.record("ch08", dict(n=n, lines_nm=nm, mean_jumps=mean_jumps, max_dev_macro_cascade=max_dev,
                            E_line_macro=E_line_macro, E_line_cascade=E_line_casc, E_line_trapped=E_line_trap,
                            beta_trap=0.05, trapped_line_nm=nm[2], p_deactivate_from_4=p_de, p_jump_from_4=p_jp,
                            n_states=atom.n_levels, n_transitions=atom.n_lines))